# Calculating Uncertainty of Measured xCO₂ Calibrated With Gas Standards
This notebook describes how QuinCe calculates the uncertainty of xCO₂ values measured in a gas analyser calibrated by periodic measurements of external gas standards. When this approach is used the reported uncertainty of the analyser can be ignored, and the characteristics of the calibration process used to calculate the uncertainty value. This will tend to be a better characterisation of the uncertainty of a given xCO₂ measurement due to the way that gas analysers can drift and/or vary over time.

## Calibrating an xCO₂ measurement
This section reviews the method that QuinCe uses to calibrate an xCO₂ measurement using external gas standards (based on the method described in Pierrot et al. 2009, doi: <a href="https://doi.org/10.1016/j.dsr2.2008.12.005">10.1016/j.dsr2.2008.12.005</a>). After this we will see how this can be extended to provide an uncertainty estimate for the measurement.


### Idenitification of relevant values
When an xCO₂ measurement is encountered, QuinCe locates the closest set of gas standard measurements taken before and after that measurement. (Quality Control activities may alter exactly which measurements are chosen, but these do not need to be considered for this explanation.) In principle, this will give us a set of measurements as follows (note that any gas standard with a zero concentration is not used in calibration):

| Timestamp           | Measurement Type | Measured xCO₂ |
|--------------------:|------------------|--------------:|
| 2026-01-12 17:42:43 | STD2             |        197.63 |
| 2026-01-12 17:44:32 | STD2             |        197.50 |
| 2026-01-12 17:46:22 | STD2             |        197.67 |
| 2026-01-12 17:50:53 | STD3             |        431.44 |
| 2026-01-12 17:52:42 | STD3             |        431.57 |
| 2026-01-12 17:54:32 | STD3             |        432.02 |
| 2026-01-12 17:59:03 | STD4             |        502.52 |
| 2026-01-12 18:00:53 | STD4             |        502.48 |
| 2026-01-12 18:02:43 | STD4             |        502.85 |
| 2026-01-12 19:30:38 | Measurement      |        423.87 |
| 2026-01-12 21:29:06 | STD2             |        197.65 |
| 2026-01-12 21:30:56 | STD2             |        197.50 |
| 2026-01-12 21:32:45 | STD2             |        197.70 |
| 2026-01-12 21:37:36 | STD3             |        431.70 |
| 2026-01-12 21:39:26 | STD3             |        431.39 |
| 2026-01-12 21:41:14 | STD3             |        431.75 |
| 2026-01-12 21:45:45 | STD4             |        502.05 |
| 2026-01-12 21:47:34 | STD4             |        502.14 |
| 2026-01-12 21:49:23 | STD4             |        502.46 |

### Calculating Standard Offsets
The `STD` values above represent measurements of the gas standards. The concentrations of those standards are as follows:

| Gas Standard | Concentration |
|--------------|--------------:|
| STD2         |       198.121 |
| STD3         |       432.817 |
| STD4         |       501.992 |

#### Prior Offsets
For each of the standard measurements prior to the measurement, we calculate the offset from the known concentration:

| Timestamp           | Measurement Type | Measured xCO₂ | Offset |
|--------------------:|------------------|--------------:|-------:|
| 2026-01-12 17:42:43 | STD2             |        197.63 | -0.491 |
| 2026-01-12 17:44:32 | STD2             |        197.50 | -0.621 |
| 2026-01-12 17:46:22 | STD2             |        197.67 | -0.451 |
| 2026-01-12 17:50:53 | STD3             |        431.44 | -1.377 |
| 2026-01-12 17:52:42 | STD3             |        431.57 | -1.247 |
| 2026-01-12 17:54:32 | STD3             |        432.02 | -0.797 |
| 2026-01-12 17:59:03 | STD4             |        502.52 |  0.528 |
| 2026-01-12 18:00:53 | STD4             |        502.48 |  0.488 |
| 2026-01-12 18:02:43 | STD4             |        502.85 |  0.858 |

We plot these offsets against the known gas standard concentrations and apply a linear regression:

In [ ]:
from datetime import datetime, timezone
from math import sqrt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tools.eval_measures import rmse

def plot_fit(xs, ys, X, fit, show_confidence):
    plot, axs = plt.subplots()
    plt.scatter(xs, ys, marker='x', color='dodgerblue')

    plt.plot([min(prior_known_concentrations), max(prior_known_concentrations)], [  fit.params.values[0], fit.params.values[1]], color='firebrick')

    pred = fit.get_prediction(X)
    pred_summary = pred.summary_frame(alpha=0.05)  # 95% confidence intervals
    ci_lower = pred_summary['mean_ci_lower']
    ci_upper = pred_summary['mean_ci_upper']

    pi_lower = pred_summary['obs_ci_lower']
    pi_upper = pred_summary['obs_ci_upper']

    plt.fill_between(xs, ci_lower, ci_upper, color='blue', alpha=0.1, label='95% CI')
    #plt.fill_between(xs, pi_lower, pi_upper, color='blue', alpha=0.2, label='95% PI')
    plt.show()

    print(f'RMSE: {rmse(ys, fit.predict(X))}')


##########################################


prior_known_concentrations = np.array([198.121, 198.121, 198.121, 432.817, 432.817, 432.817, 501.992, 501.992, 501.992])
prior_offsets = np.array([-0.491, -0.621, -0.451, -1.377, -1.247, -0.797, 0.528, 0.488, 0.858])

X = pd.DataFrame({'intercept':np.ones(len(prior_known_concentrations)), 'slope': prior_known_concentrations})
prior_fit = sm.OLS(prior_offsets, X).fit()
plot_fit(prior_known_concentrations, prior_offsets, X, prior_fit, True)


#### Post Offsets
Now we perform the same process for the post-measurement gas standards.

| Timestamp           | Measurement Type | Measured xCO₂ | Offset |
|--------------------:|------------------|--------------:|-------:|
| 2026-01-12 21:29:06 | STD2             |        197.65 | -0.471 |
| 2026-01-12 21:30:56 | STD2             |        197.50 | -0.621 |
| 2026-01-12 21:32:45 | STD2             |        197.70 | -0.421 |
| 2026-01-12 21:37:36 | STD3             |        431.70 | -1.117 |
| 2026-01-12 21:39:26 | STD3             |        431.39 | -1.427 |
| 2026-01-12 21:41:14 | STD3             |        431.75 | -1.067 |
| 2026-01-12 21:45:45 | STD4             |        502.05 |  0.058 |
| 2026-01-12 21:47:34 | STD4             |        502.14 |  0.148 |
| 2026-01-12 21:49:23 | STD4             |        502.46 |  0.468 |


In [ ]:
post_known_concentrations = np.array([198.121, 198.121, 198.121, 432.817, 432.817, 432.817, 501.992, 501.992, 501.992])
post_offsets = np.array([-0.471, -0.621, -0.421, -1.117, -1.427, -1.067, 0.058, 0.148, 0.468])
X = pd.DataFrame({'intercept':np.ones(len(post_known_concentrations)), 'slope': post_known_concentrations})
post_fit = sm.OLS(post_offsets, X).fit()
print(post_fit.summary())
plot_fit(post_known_concentrations, post_offsets, X, prior_fit, True)


### Get the standard offset for the measurement
Our measured value is 423.87 μmol/mol (or ppm). For each of the prior and post assessments, we calculate the offset for that measurement value, with the uncertainty being the RMSE of the linear fit across the standards. At the same time, calculate the mean timestamp of the prior and post standard measurements for later.

In [ ]:
# Times have been manually converted to UNIX timestamps
prior_times = [1768239763, 1768239872, 1768239982, 1768240253, 1768240362, 1768240472, 1768240743, 1768240853, 1768240963]
mean_prior_time = datetime.fromtimestamp(np.mean(prior_times), timezone.utc)

post_times = [1768253346, 1768253456, 1768253565, 1768253856, 1768253966, 1768254074, 1768254345, 1768254454, 1768254563]
mean_post_time = datetime.fromtimestamp(np.mean(post_times), timezone.utc)

prior_post_times = [mean_prior_time, mean_post_time]

prior_offset = prior_fit.get_prediction([1, 423.87])
post_offset = post_fit.get_prediction([1, 423.87])

# Build the scatter plot entries
values = [prior_offset.predicted_mean[0], post_offset.predicted_mean[0]]
min_errors = [values[0] - prior_offset.conf_int()[0][0], values[1] - post_offset.conf_int()[0][0]]
max_errors = [prior_offset.conf_int()[0][1] - values[0], post_offset.conf_int()[0][1] - values[1]]
plot, axs = plt.subplots()
plt.errorbar(prior_post_times, values, [min_errors, max_errors], marker='o')
plt.show()


#print(f'Prior offset: Mean time {mean_prior_time.strftime('%Y-%m-%d %H:%M:%S')}, Offset {prior_offset:.4f}±{prior_rmse:.4f}')
#print(f'Post  offset: Mean time {mean_post_time.strftime('%Y-%m-%d %H:%M:%S')}, Offset {post_offset:.4f}±{post_rmse:.4f}')
